# Do Chatbot ao Agente: Construindo Sistemas de IA com Ferramentas

Neste notebook, uma aplicação com LLM evolui, passo a passo, até um **agente** que planeja, usa ferramentas e mostra o
que fez. Em cada etapa você roda uma célula curta e vê o resultado.

| Etapa | O que você vai ver |
|---|---|
| 1. Ambiente | o repositório e as dependências prontos |
| 2. Conexão | o "chatbot": uma pergunta, uma resposta |
| 3. Montar o agente | as peças (modelo, ferramentas, memória, tracer) e o loop |
| 4. Agente sem ferramentas | ele planeja e responde com o que o modelo já sabe |
| 5. A ferramenta certa? | sem acesso ao documento, ele avisa em vez de inventar |
| 6. RAG | com uma ferramenta de busca, ele responde citando o PPC do curso |
| 7. Bastidores | o tracing (passos, tokens, tempo) e a memória temporária |
| 8. Avaliação | evals: conferir automaticamente se o agente se comporta bem |
| 9. Limitações | contexto, limites da API, alucinação e confiabilidade |

**Antes de começar (Colab):** no painel **Secrets** (ícone de chave 🔑 à esquerda), crie o secret `GROQ_API_KEY` com a
**sua** chave do [console do Groq](https://console.groq.com/keys) e ative **"Acesso ao notebook"**. Opcional:
`GROQ_MODEL` para trocar o modelo. Nunca cole a chave numa célula.

> Cada passo do agente é **uma chamada à API** e conta para os limites do plano gratuito.

## 1. Preparar o ambiente

No Colab, clona o repositório e instala as dependências, inclusive as da busca do RAG (cerca de 1 min). Localmente
(VS Code/Jupyter com o kernel do `.venv`), usa a própria pasta do repositório. O código está recolhido: não é o foco do
workshop, mas você pode abri-lo.

> Se o repositório for atualizado **depois** de importar o pacote, reinicie a sessão (*Ambiente de execução → Reiniciar
> sessão*) e rode as células de novo.

In [ ]:
# @title Preparar o ambiente { display-mode: "form" }
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO_URL = "https://github.com/pedroop32/workshop-agentes.git"
REPO_DIR = "/content/workshop-agentes"
IN_COLAB = "google.colab" in sys.modules


def sh(*cmd: str) -> None:
    """Roda um comando; mostra a saída só se der erro."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout, result.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", REPO_URL, REPO_DIR)
    # requirements-rag.txt inclui o requirements.txt (o agente) e acrescenta a busca do RAG
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-rag.txt")
else:
    # Local: a pasta do repositório é a atual ou uma acima dela.
    local = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_agentes").is_dir()), None)
    assert local, "Abra o notebook de dentro da pasta do repositório (onde está workshop_agentes/)."
    REPO_DIR = str(local)
    faltam = [nome for nome in ("numpy", "pypdf", "fastembed", "rank_bm25") if find_spec(nome) is None]
    if faltam:
        print(f"Para o RAG (seção 6): .venv/bin/python -m pip install -r requirements-rag.txt (faltam: {', '.join(faltam)})")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Ambiente pronto ({'Colab' if IN_COLAB else 'local'}): {REPO_DIR}")

## 2. Testar a conexão: o "chatbot"

A aplicação mais simples com LLM é esta: mandamos uma mensagem e recebemos uma resposta. A API é **stateless**: ela
não lembra de nada entre as chamadas. Quem guarda o histórico é o nosso código (você vai ver isso na seção 7).

A chamada passa pelo mesmo **retry com limite** do agente: se o plano gratuito estourar o limite por minuto (erro 429),
ela espera e tenta de novo. Se falhar, aparece o que fazer (chave, modelo, limite), nunca a mensagem bruta do erro.

In [ ]:
from workshop_agentes.agent.config import DEFAULT_MODEL, AgentConfig, get_secret
from workshop_agentes.llm import build_llm
from workshop_agentes.visual import check_connection

GROQ_API_KEY = get_secret("GROQ_API_KEY")  # Colab Secrets, variável de ambiente ou .env; nunca é impressa
assert GROQ_API_KEY, "Crie o secret GROQ_API_KEY (ícone de chave) e ative o acesso do notebook."
MODEL = get_secret("GROQ_MODEL", DEFAULT_MODEL)

config = AgentConfig()  # até 8 passos; a 1ª chamada é obrigada a planejar; mostra as notas ao vivo
llm = build_llm(MODEL, GROQ_API_KEY, temperature=config.temperature)
print(f"Modelo: {MODEL}")
check_connection(llm, GROQ_API_KEY)

## 3. Montar o agente

Um **agente** é um LLM dentro de um loop. O modelo não executa nada: ele só **pede** ferramentas. Quem executa é o
nosso código, que devolve o resultado ao modelo, e o ciclo se repete até ele responder em texto.

```
tarefa ─► modelo ─► pediu ferramenta? ─ sim ─► nosso código executa ─► resultado volta ao modelo ─┐
            ▲              │                                                                     │
            │              └─ não ─► resposta final em texto (fim)                               │
            └────────────────────────────────────────────────────────────────────────────────────┘
```

As peças são criadas separadamente e **injetadas** no agente: trocar o modelo, as ferramentas ou a memória não muda o
loop.

| Peça | Papel |
|---|---|
| `llm` | o modelo (Groq) |
| `ToolRegistry` | as ferramentas que o modelo pode pedir |
| `WorkingMemory` | o histórico da execução, reenviado a cada passo |
| `Tracer` | mede cada passo: tokens, tempo e ferramentas |
| `AgentConfig` | limite de passos, system prompt e a 1ª ferramenta obrigatória |

Por enquanto, a única ferramenta é o `update_plan`: o modelo **registra o plano** (o que já sabe, o plano e o próximo
passo) e escreve uma nota curta para você. Ela não busca nada nem age no mundo.

In [ ]:
from workshop_agentes.agent.loop import Agent
from workshop_agentes.agent.memory import WorkingMemory
from workshop_agentes.tools.core import update_plan
from workshop_agentes.tools.registry import ToolRegistry
from workshop_agentes.tracing.tracer import Tracer
from workshop_agentes.visual import show_result, show_tools


def montar_agente(*ferramentas):
    """Um agente novo com as ferramentas dadas e as mesmas peças (modelo e configuração)."""
    registry = ToolRegistry()
    for ferramenta in ferramentas:
        registry.register(ferramenta)
    return Agent(llm=llm, registry=registry, memory=WorkingMemory(config.system_prompt), tracer=Tracer(), config=config)


agente = montar_agente(update_plan)
show_tools(agente)

## 4. Executar o agente (sem ferramentas de ação)

O agente recebe a tarefa, **planeja** (a 1ª chamada é obrigada a usar `update_plan`) e responde com o que o modelo já
sabe. As linhas com `→` aparecem **durante** a execução: é a nota que o modelo escreveu para você. Aqui, um run típico
tem **2 passos**: o plano e a resposta em texto.

<details>
<summary>Por que um run termina (motivo de parada)</summary>

| motivo | significado |
|---|---|
| `no_tool_call` | o modelo respondeu só com texto: é a resposta final |
| `length` | a resposta foi cortada pelo limite de tokens (está incompleta) |
| `max_steps` | acabaram os passos sem resposta final |
| `error` | a chamada ao modelo falhou |

</details>

Se aparecer `[retentando em …: RateLimitError …]`, o agente bateu no limite por minuto do plano gratuito e está
esperando a cota voltar. Não precisa fazer nada.

In [ ]:
TAREFA = "Em até 3 frases, explique a diferença entre um chatbot e um agente de IA."

resultado = agente.run(TAREFA)
show_result(resultado)

## 5. O agente tem a ferramenta certa para a tarefa?

Um agente só faz o que as suas ferramentas permitem. Agora a pergunta está **num documento**: o PPC (Projeto
Pedagógico) do curso de Ciência da Computação, que o modelo nunca viu. Na tabela de ferramentas, nada lê documentos.

O system prompt manda o modelo **avisar quando não puder** fazer a tarefa (faltam documentos, dados em tempo real, uma
ação como enviar e-mail), em vez de inventar. Se mesmo assim ele inventar um número, isso também é uma lição: modelos
erram com confiança, e é para isso que servem as avaliações (seção 8).

In [ ]:
PERGUNTA = "Segundo o PPC do curso, qual é a carga horária total e quantas horas de atividades complementares são exigidas?"

show_tools(agente)  # só o update_plan: nenhuma ferramenta lê documentos
sem_rag = agente.run(PERGUNTA)
show_result(sem_rag)

## 6. RAG: dando ao agente uma ferramenta de busca

**RAG** (*Retrieval-Augmented Generation*): antes de responder, o agente **busca** trechos relevantes nos documentos,
recebe esses trechos no contexto e responde **com base neles**, citando a fonte.

```
pergunta ─► modelo pede search_documents("carga horária total") ─► busca nos PDFs (local) ─┐
                                                                                            │
resposta citando arquivo e página ◄─ modelo lê os trechos ◄─ trechos com arquivo e página ◄─┘
```

- A busca roda **aqui, sem chamar a API** (embeddings + palavras-chave): só os trechos escolhidos vão ao modelo.
- O PDF de exemplo é o **PPC 2024 de Ciência da Computação** (`documentos/PPC2024BCC.pdf`, 201 páginas). Para usar os
  seus, coloque PDFs com texto selecionável na pasta `documentos/`.
- Antes, o PDF é **indexado**: dividido em trechos de ~1000 caracteres, e cada trecho vira um vetor. Na 1ª vez leva
  cerca de 1 min (baixa o modelo de busca, ~120 MB); depois usa o cache.

In [ ]:
import time

from workshop_agentes.rag import PdfRag
from workshop_agentes.tools.documents import make_search_documents
from workshop_agentes.visual import show_index, show_passages

motor = PdfRag()  # os PDFs da pasta documentos/
inicio = time.perf_counter()
motor.refresh()  # indexa (ou lê o índice do cache)
show_index(motor, time.perf_counter() - inicio)

Agora o agente ganha a ferramenta `search_documents` e recebe **a mesma pergunta**. Durante a execução, além da nota
`→`, aparece a linha `[search_documents {...}]`: a consulta que o modelo escolheu e o tamanho do resultado. Depois vêm
os trechos que a busca devolveu ao modelo.

In [ ]:
agente_rag = montar_agente(update_plan, make_search_documents(motor))
show_tools(agente_rag)

com_rag = agente_rag.run(PERGUNTA)
show_result(com_rag)
show_passages(com_rag)

## 7. Bastidores: tracing e memória temporária

O que aconteceu **durante** o run com RAG:

- **Tracing:** cada passo é medido (*spans*: run → passo → chamada ao modelo ou de ferramenta). A coluna **Entrada** é o
  contexto, o histórico inteiro que o modelo recebeu naquele passo. Repare como ela **cresce** depois da busca: os
  trechos entram no histórico e são reenviados a cada chamada. Contexto custa tokens, tempo e cota.
- **Memória temporária:** as mensagens do run, na ordem em que o modelo as recebe: o system prompt, a pergunta, cada
  pedido de ferramenta seguido do seu resultado, e a resposta final. Ela existe **só durante o run**: o próximo `run()`
  começa do zero.

In [ ]:
from workshop_agentes.visual import show_trace

show_trace(com_rag)

In [ ]:
from workshop_agentes.visual import show_memory

show_memory(com_rag)
print("Toda chamada de ferramenta tem o seu resultado logo depois:", agente_rag.memory.check_invariant())

## 8. Avaliação (evals)

Um agente pode responder diferente a cada execução, e uma mudança no prompt ou no modelo pode quebrar o que funcionava.
**Evals** conferem isso automaticamente: um *golden set* de casos (tarefa + o que esperar) e **checks determinísticos**
em cada run, lidos do tracing. O run concluiu? Usou (ou evitou) a ferramenta certa? A resposta tem o termo esperado?
Em quantos passos? Opcionalmente, um **juiz LLM** compara a resposta com uma rubrica.

Os três casos abaixo resumem o workshop:

| caso | o que confere |
|---|---|
| `ppc-carga-horaria` | usou `search_documents` e a resposta tem 3472 h (RAG) |
| `aritmetica` | 17 × 23 = 391 **sem** usar a busca (não chamar ferramenta à toa) |
| `acao-sem-ferramenta` | pedido para enviar um e-mail: avisa que não pode (limitação) |

Custo: cerca de 7 chamadas; com `JUIZ = True`, +1 por caso com rubrica. Os casos ficam em
`workshop_agentes/evals/golden_set.json`.

In [ ]:
from workshop_agentes.evals.cases import load_golden_set
from workshop_agentes.evals.judge import Judge
from workshop_agentes.evals.runner import run_suite
from workshop_agentes.visual import show_evals

CASOS = ["ppc-carga-horaria", "aritmetica", "acao-sem-ferramenta"]  # None roda todos os casos do golden set
JUIZ = False  # True: um LLM avalia a rubrica de cada caso (+1 chamada por caso)

golden_set = {caso.id: caso for caso in load_golden_set()}
casos = list(golden_set.values()) if CASOS is None else [golden_set[id_] for id_ in CASOS]  # na ordem de CASOS
juiz = Judge(build_llm(MODEL, GROQ_API_KEY, temperature=0.0)) if JUIZ else None
avaliacao = run_suite(agente_rag, casos, judge=juiz)
show_evals(avaliacao)

## 9. Limitações e confiabilidade

- **Contexto tem custo.** Cada passo reenvia o histórico inteiro (seção 7), e cada busca acrescenta ~1.500 a 2.000
  tokens. No plano gratuito, os limites de tokens por minuto (de entrada, ITPM, e de saída, OTPM) aparecem como
  `[retentando em …]`: o agente espera e tenta de novo, e o tracing registra a espera.
- **Alucinação.** Sem a ferramenta, um modelo pode inventar um número com confiança. Com RAG, a resposta cita arquivo
  e página, e dá para conferir. Mesmo assim, a busca pode trazer o trecho errado (tabelas mal extraídas do PDF,
  páginas parecidas).
- **Trechos são dados, não instruções.** Um PDF pode conter algo como "ignore as instruções anteriores" (*prompt
  injection*). A descrição da ferramenta manda o modelo tratar os trechos só como dados, e o nosso código nunca executa
  o que vem neles.
- **Limites de execução.** O `max_steps` impede loops sem fim. O motivo de parada (`no_tool_call`, `length`,
  `max_steps`, `error`) diz por que o run terminou. As retentativas também têm limite (cerca de 1 min por chamada).
- **Não determinismo.** A mesma pergunta pode gerar respostas e caminhos diferentes (uma busca ou duas). Por isso
  avaliamos com vários casos e repetimos as execuções, em vez de confiar num teste só.

Para ir além, na linha de comando: `python -m workshop_agentes.evals --dry-run` mostra o que rodaria, sem usar a API, e
`python -m workshop_agentes.evals --judge` roda a suíte completa com o juiz.